## Load Radar Config

Carrega o **radar IMDM de cada edição** (mantido pelo especialista em `config/radar/radar_imdm.xlsx`) para tabelas Delta, que passam a ser a fonte das dimensões, itens, códigos por edição e pontuações do IMDM.

- 1. Lê os CSVs de `config/radar/csv/` — gerados pelo `validar_radar.py` a partir da planilha e versionados no **mesmo commit** do código (o Git folder já os traz; nada é copiado à mão)
- 2. Valida o radar da edição do job (`reference_year`) com as **mesmas regras** do `validar_radar.py` (o módulo é importado, não duplicado) e contra a **silver da edição** (`reference_year`): toda variável existe com dados e toda resposta dos itens escalonados tem pontuação. Qualquer ERRO interrompe o run **antes** de gravar
- 3. Grava `config.radar_*` (catálogo inteiro, overwrite — o histórico de versões fica no `DESCRIBE HISTORY` de cada tabela) com o SHA-256 da planilha em todas as linhas
- 4. Monta, a partir das tabelas, a lista de itens da edição **no mesmo formato** do `imdm_items` do `silver_to_gold` (Step 3) e registra a linha `radar` no ledger

> Fluxo de mudança do radar: especialista edita a planilha → `python config/radar/validar_radar.py --dados <ano>=<base>.xlsx` → commit (planilha + CSVs) → PR → job. O especialista decide **o quê**; as regras de cálculo (**como**) continuam no código.

In [0]:
%run ../environment/env_variables

In [0]:
%run ../environment/env_functions

## Step 1 - Ler o catálogo versionado no repositório

- O notebook roda dentro do Git folder (ou do bundle): o catálogo está em `../config/radar`, na mesma versão do código
- O `validar_radar.py` é importado como módulo — a regra de validação vive em um lugar só (local e Databricks)

In [0]:
import hashlib
import importlib
import json
import sys
from pathlib import Path

import pandas as pd
from pyspark.sql import functions as F

# Diretório do notebook = pipeline/ ; o catálogo fica em ../config/radar (mesmo commit do código)
radar_dir = (Path.cwd().parent / "config" / "radar").resolve()
csv_dir = radar_dir / "csv"
if not (csv_dir / "radar_versao.json").exists():
    raise FileNotFoundError(
        f"Catálogo do radar não encontrado em {csv_dir}. Rode localmente "
        "'python config/radar/validar_radar.py' e faça commit da planilha + CSVs."
    )

if str(radar_dir) not in sys.path:
    sys.path.insert(0, str(radar_dir))
import validar_radar as vr
importlib.reload(vr)   # pega a versão do repo mesmo se o módulo já foi importado na sessão

cat = vr.ler_csvs(csv_dir)
versao = json.loads((csv_dir / "radar_versao.json").read_text(encoding="utf-8"))

# Hash do que está sendo CARREGADO (os CSVs), além do hash da planilha que os gerou
h = hashlib.sha256()
for arq in sorted(vr.ABAS.values()):
    h.update((csv_dir / arq).read_bytes())
sha256_csvs = h.hexdigest()
sha256_planilha = versao["sha256_planilha"]

print(f"Catálogo: {csv_dir}")
print(f"Planilha {versao['planilha']} sha256={sha256_planilha[:12]}… | CSVs sha256={sha256_csvs[:12]}… | gerado em {versao['gerado_em']}")
for aba, df in cat.items():
    print(f"  {aba:<16} {len(df):>4} linhas")


## Step 2 - Validar (estrutura + silver da edição)

- Estrutura: ids únicos, regra e exceção do vocabulário fechado, item presente com variáveis, ausente/excluído com motivo, nenhuma variável em dois itens, dimensão do IMDM com pelo menos um item, faixas de nível sem buraco
- Silver da edição: toda variável dos itens presentes tem dados em `reference_year`; toda resposta observada nos itens escalonados de 1 variável tem pontuação (rótulo novo do IBGE não vira 0 silencioso)
- Fail-fast: qualquer ERRO interrompe aqui, antes de gravar

In [0]:
silver_df = spark.table(f"{catalog_name}.{silver_schema}.munic_clean_data").filter(F.col("ano_edicao") == reference_year)

edicoes_catalogo = vr.edicoes(cat)
if reference_year not in edicoes_catalogo:
    raise ValueError(
        f"A edição {reference_year} não tem radar na planilha (edições com radar: {edicoes_catalogo}). "
        "O especialista precisa acrescentar as linhas da edição nas abas Dimensoes, Itens e Niveis."
    )

# Colunas com dados na edição (mesma função usada pelo silver_to_gold)
todas_indicadoras = [c for c in silver_df.columns if c.startswith(("mtic", "mgov"))]
com_dados = set(colunas_com_dados(silver_df, todas_indicadoras))

# Valores distintos só das variáveis pontuadas por rótulo (escalonados de 1 variável)
itens_ano = cat["Itens"][cat["Itens"]["edicao"] == str(reference_year)]
vars_rotulo = sorted({vr.variaveis(r.variaveis)[0] for r in itens_ano.itertuples()
                      if r.regra == "CATEGORICO_ESCALONADO" and len(vr.variaveis(r.variaveis)) == 1} & com_dados)
valores = {}
if vars_rotulo:
    linha = silver_df.agg(*[F.collect_set(c).alias(c) for c in vars_rotulo]).collect()[0].asDict()
    valores = {c: {str(v).strip() for v in vs} for c, vs in linha.items()}

erros, avisos = vr.validar(cat, {reference_year: {"colunas": com_dados, "valores": valores}}, so_edicao=reference_year)
for a in avisos:
    print(f"[AVISO] {a}")
for e in erros:
    print(f"[ERRO]  {e}")
if erros:
    raise AssertionError(f"Catálogo do radar reprovado: {len(erros)} erro(s) — nada foi gravado. Corrija a planilha, gere os CSVs e faça novo commit.")

res = vr.resumo(cat)[str(reference_year)]
print(f"OK: radar {reference_year} — {res['itens']} itens em {len(res['dimensoes'])} dimensões:")
for d in res["dimensoes"]:
    print(f"  {d['ordem']}. {d['dimensao']} — {d['itens']} itens | entra no IMDM: {d['entra_no_imdm']}")


## Step 3 - Gravar `config.radar_*`

Cada edição tem o seu radar; todas as tabelas têm a coluna `edicao` e nenhuma edição referencia outra.

| Tabela | Uma linha por | Uso |
|---|---|---|
| `radar_dimensao` | edição × dimensão | nome, ordem, peso, entra no IMDM (eixos do radar no dashboard) |
| `radar_item` | edição × item | dimensão, regra, pontuação (como digitada), exceção |
| `radar_variavel` | edição × item × variável | o join do `silver_to_gold`: `cod_variable` → item, dimensão, regra, âncora |
| `radar_pontuacao` | edição × item × variável × resposta | pontos por rótulo do IBGE (forma longa da coluna pontuacao) |
| `radar_nivel` | edição × nível | faixas do `nivel_maturidade_mmd` |

Catálogo inteiro em overwrite: as versões anteriores ficam no `DESCRIBE HISTORY`, e toda linha carrega `radar_sha256` (planilha) e `carregado_em`.

In [0]:
config_schema = "config"
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{config_schema} "
    "COMMENT 'Parametrização mantida pelo negócio (radar IMDM). Escrita só pelo job load_radar_config.'"
)

derivadas = vr.tabelas_derivadas(cat)
dimensoes = cat["Dimensoes"].drop(columns="linha_planilha").assign(
    dimensao_codigo=lambda d: d["dimensao"].map(vr.codigo_dimensao))
saidas = {
    "radar_dimensao": (dimensoes, "Dimensões do radar IMDM por edição (eixos do radar)."),
    "radar_item": (cat["Itens"].drop(columns="linha_planilha"), "Itens do radar por edição: dimensão, códigos, regra e pontuação."),
    "radar_variavel": (derivadas["variaveis"], "Edição × item × variável do IBGE; e_ancora = variável que recebe os pontos."),
    "radar_pontuacao": (derivadas["pontuacao"], "Pontos por resposta do IBGE (escalonados e escadas), por edição."),
    "radar_nivel": (cat["Niveis"].drop(columns="linha_planilha"), "Faixas do nível de maturidade por edição (0 a 100)."),
}
inteiros = {"edicao", "ordem", "nivel"}
decimais = {"peso", "minimo", "maximo", "pontos"}

carregado_em = pd.Timestamp.now(tz="UTC").tz_localize(None)
for nome, (pdf, descricao) in saidas.items():
    pdf = pdf.copy()
    for c in pdf.columns:
        if c in inteiros:
            pdf[c] = pdf[c].astype(int)
        elif c in decimais:
            pdf[c] = pdf[c].map(vr.numero).astype(float)
    pdf["radar_sha256"] = sha256_planilha
    pdf["carregado_em"] = carregado_em
    tabela = f"{catalog_name}.{config_schema}.{nome}"
    (spark.createDataFrame(pdf).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela))
    spark.sql(f"COMMENT ON TABLE {tabela} IS '{descricao} Origem: config/radar/radar_imdm.xlsx (sha256 {sha256_planilha[:12]}).'")
    print(f"Gravada: {tabela} ({len(pdf)} linhas)")


## Step 4 - Conferência e ledger

- Relê as tabelas gravadas e monta os itens da edição no **formato do `imdm_items`** do `silver_to_gold`: `(dimensao, âncora, regra, variáveis-membro)` — é o que o Step 3 da gold passa a ler no lugar da lista fixa
- Registra a linha `radar` × edição no ledger `pipeline_manifest` (versão do radar usada pela edição)

In [0]:
def itens_do_catalogo(ano):
    """Itens do radar da edição no formato do imdm_items do silver_to_gold:
    (dimensao_codigo, variável-âncora, regra, variáveis-membro)."""
    rv = (spark.table(f"{catalog_name}.config.radar_variavel")
          .filter(F.col("edicao") == ano)
          .orderBy("item", "ordem")
          .toPandas())
    return [(g["dimensao_codigo"].iloc[0], g["cod_variable"].iloc[0], g["regra"].iloc[0], g["cod_variable"].tolist())
            for _, g in rv.groupby("item", sort=False)]

imdm_items_catalogo = itens_do_catalogo(reference_year)
from collections import Counter
contagem = Counter(d for d, _, _, _ in imdm_items_catalogo)
esperado = {vr.codigo_dimensao(d["dimensao"]): d["itens"] for d in res["dimensoes"] if d["itens"]}

check_igual(len(imdm_items_catalogo), res["itens"], f"radar {reference_year}: itens gravados")
check_igual(dict(sorted(contagem.items())), dict(sorted(esperado.items())), f"radar {reference_year}: itens por dimensão")
resumo_validacoes()

itens_ano = derivadas["variaveis"][derivadas["variaveis"]["edicao"] == reference_year]
gravar_linha_manifest(
    etapa="radar",
    ano_edicao=reference_year,
    tabela_destino=";".join(f"{catalog_name}.{config_schema}.{n}" for n in saidas),
    sha256_xlsx=sha256_planilha,
    linhas=res["itens"],
    indicadores=int(itens_ano["cod_variable"].nunique()),
    detalhes={
        "dimensoes": " | ".join(f"{d['ordem']}. {d['dimensao']} ({d['itens']})" for d in res["dimensoes"]),
        "dimensoes_no_imdm": str(sum(d["entra_no_imdm"].lower() == "sim" for d in res["dimensoes"])),
        "sha256_csvs": sha256_csvs,
        "csv_gerado_em": versao["gerado_em"],
        "avisos": str(len(avisos)),
    },
)
display(pd.DataFrame(imdm_items_catalogo, columns=["dimensao", "ancora", "regra", "variaveis"]))
